# Ingest `sprints` folder

1. Read the file using spark dataframe reader API
2. Define and enforce schema (preserve the nested structure)
3. Add metadata columns
    - Source File
    - Ingestion Timestamp
4. Write to Bronze delta table

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/02.bronze-helpers

## Define source_file and table_name

In [0]:
source_file = f'{landing_folder_path}/sprints'
table_name = f'{catalog_name}.{bronze_schema}.sprints'

## Step 1 - Read JSON file using the dataframe reader API

In [0]:
# Define the schema
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType, FloatType

sprints_schema = StructType([
    StructField('date', DateType()),
    StructField('raceName', StringType()),
    StructField('round', IntegerType()),
    StructField('season', IntegerType()),
    StructField('url', StringType()),
    StructField('constructorId', StringType()),
    StructField('driverId', StringType()),
    StructField('grid', IntegerType()),
    StructField('laps', IntegerType()),
    StructField('number', IntegerType()),
    StructField('points', FloatType()),
    StructField('position', IntegerType()),
    StructField('positionText', StringType()),
    StructField('status', StringType()),
])


In [0]:
# Read data from the constructors file
sprints_df = (spark.read
                   .format('json')
                   .schema(sprints_schema)
                   .option('mode', 'FAILFAST')
                   .option('multiline', True)
                   .load(source_file)
                   )

In [0]:
display(sprints_df)

## Step 2 - Add Metadata Columns

In [0]:
sprints_final_df = add_ingetion_metadata(sprints_df)

In [0]:
display(sprints_final_df)

## Step 3 - Write to bronze delta table

In [0]:
(
    sprints_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))

In [0]:
%sql
SELECT season, COUNT(*)
FROM formula1.bronze.sprints
GROUP BY season
ORDER BY season